In [0]:
from datetime import datetime, timedelta
from pyspark.sql import functions as F
import random
import uuid
import json
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    TimestampType,
    DateType,
    IntegerType,
    DoubleType,
    BooleanType
)


In [0]:
dbutils.widgets.text("CATALOG", "sales")
CATALOG = dbutils.widgets.get("CATALOG")

dbutils.widgets.text("raw_schema", "raw_layer")
raw_schema = dbutils.widgets.get("raw_schema")

spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{raw_schema}")

In [0]:
RAW_PATH = f"/Volumes/{CATALOG}/{raw_schema}/sales_generated_data"

AUDIT_TABLE = f"{CATALOG}.{raw_schema}.sales_generation_audit"

NUM_RECORDS = 100

RUN_ID = str(uuid.uuid4())
RUN_TIMESTAMP = datetime.now()

# Five independent source systems.
SOURCE_SYSTEMS = [
    "POS",
    "E_COMMERCE",
    "MOBILE_APP",
    "RETAIL_STORE",
    "MARKETPLACE"
]

random.seed()

In [0]:
locations = [
    ("India","Asia","Delhi","Delhi","110001"),
    ("India","Asia","Mumbai","Maharashtra","400001"),
    ("India","Asia","Pune","Maharashtra","411001"),
    ("India","Asia","Bengaluru","Karnataka","560001"),
    ("India","Asia","Chennai","Tamil Nadu","600001"),
    ("India","Asia","Hyderabad","Telangana","500001"),
    ("India","Asia","Kolkata","West Bengal","700001"),
    ("India","Asia","Ahmedabad","Gujarat","380001"),
    ("India","Asia","Surat","Gujarat","395003"),
    ("India","Asia","Jaipur","Rajasthan","302001"),
    ("India","Asia","Lucknow","Uttar Pradesh","226001"),
    ("India","Asia","Kanpur","Uttar Pradesh","208001"),
    ("India","Asia","Patna","Bihar","800001"),
    ("India","Asia","Gaya","Bihar","823001"),
    ("India","Asia","Ranchi","Jharkhand","834001"),
    ("India","Asia","Bhopal","Madhya Pradesh","462001"),
    ("India","Asia","Indore","Madhya Pradesh","452001"),
    ("India","Asia","Nagpur","Maharashtra","440001"),
    ("India","Asia","Nashik","Maharashtra","422001"),
    ("India","Asia","Chandigarh","Chandigarh","160001"),
    ("India","Asia","Ludhiana","Punjab","141001"),
    ("India","Asia","Amritsar","Punjab","143001"),
    ("India","Asia","Dehradun","Uttarakhand","248001"),
    ("India","Asia","Srinagar","Jammu Kashmir","190001"),
    ("India","Asia","Guwahati","Assam","781001"),
    ("India","Asia","Bhubaneswar","Odisha","751001"),
    ("India","Asia","Kochi","Kerala","682001"),
    ("India","Asia","Thiruvananthapuram","Kerala","695001"),
    ("India","Asia","Visakhapatnam","Andhra Pradesh","530001"),
    ("India","Asia","Vijayawada","Andhra Pradesh","520001"),
    ("India","Asia","Mysuru","Karnataka","570001"),
    ("India","Asia","Coimbatore","Tamil Nadu","641001"),
    ("India","Asia","Madurai","Tamil Nadu","625001"),
    ("India","Asia","Raipur","Chhattisgarh","492001"),
    ("India","Asia","Jamshedpur","Jharkhand","831001"),
    ("India","Asia","Varanasi","Uttar Pradesh","221001"),
    ("India","Asia","Agra","Uttar Pradesh","282001"),
    ("India","Asia","Meerut","Uttar Pradesh","250001"),
    ("India","Asia","Prayagraj","Uttar Pradesh","211001"),
    ("India","Asia","Muzaffarpur","Bihar","842001"),
    ("India","Asia","Darbhanga","Bihar","846001"),

    ("USA","North America","New York","New York","10001"),
    ("USA","North America","Los Angeles","California","90001"),
    ("USA","North America","Chicago","Illinois","60601"),
    ("USA","North America","Houston","Texas","77001"),
    ("USA","North America","Phoenix","Arizona","85001"),
    ("USA","North America","Dallas","Texas","75201"),

    ("UK","Europe","London","England","SW1A"),
    ("UK","Europe","Manchester","England","M1"),
    ("UK","Europe","Birmingham","England","B1"),

    ("Canada","North America","Toronto","Ontario","M5V"),
    ("Canada","North America","Vancouver","British Columbia","V5K"),

    ("Australia","Oceania","Sydney","New South Wales","2000"),
    ("Australia","Oceania","Melbourne","Victoria","3000"),

    ("Germany","Europe","Berlin","Berlin","10115"),
    ("Germany","Europe","Munich","Bavaria","80331"),

    ("France","Europe","Paris","Île-de-France","75001"),
    ("France","Europe","Lyon","Auvergne","69001"),

    ("Japan","Asia","Tokyo","Tokyo","1000001"),
    ("Japan","Asia","Osaka","Osaka","5300001"),

    ("China","Asia","Beijing","Beijing","100000"),
    ("China","Asia","Shanghai","Shanghai","200000"),

    ("UAE","Middle East","Dubai","Dubai","00000"),
    ("Singapore","Asia","Singapore","Singapore","018956"),

    ("South Africa","Africa","Johannesburg","Gauteng","2000"),
    ("Brazil","South America","Sao Paulo","Sao Paulo","01000"),
    ("Italy","Europe","Rome","Lazio","00100"),
    ("Spain","Europe","Madrid","Madrid","28001"),
    ("Mexico","North America","Mexico City","Mexico","01000"),
    ("South Korea","Asia","Seoul","Seoul","04524"),
    ("Thailand","Asia","Bangkok","Bangkok","10100"),
    ("Malaysia","Asia","Kuala Lumpur","Selangor","50000"),
    ("Indonesia","Asia","Jakarta","Jakarta","10110"),
    ("Vietnam","Asia","Ho Chi Minh City","Ho Chi Minh","700000"),
    ("Saudi Arabia","Middle East","Riyadh","Riyadh","11564"),
    ("Egypt","Africa","Cairo","Cairo","11511"),
    ("Nigeria","Africa","Lagos","Lagos","100001"),
    ("Kenya","Africa","Nairobi","Nairobi","00100")
]

In [0]:
product_categories = {
    "Electronics": [
        "Laptop", "Desktop PC", "Monitor", "Keyboard", "Mouse", "Webcam",
        "Printer", "Tablet", "Smartphone", "Smart Watch", "Headphones",
        "Bluetooth Speaker", "Power Bank", "Router", "External SSD", "External HDD"
    ],

    "Fashion": [
        "T-Shirt", "Shirt", "Jeans", "Trousers", "Jacket", "Sweater",
        "Hoodie", "Dress", "Saree", "Kurta", "Shoes", "Sneakers",
        "Sandals", "Boots", "Cap", "Sunglasses"
    ],

    "Home": [
        "Refrigerator", "Washing Machine", "Microwave", "Air Conditioner",
        "Air Purifier", "Vacuum Cleaner", "Coffee Maker", "Electric Kettle",
        "Toaster", "Mixer Grinder", "Iron", "Room Heater"
    ],

    "Beauty": [
        "Face Wash", "Moisturizer", "Sunscreen", "Shampoo", "Conditioner",
        "Perfume", "Body Lotion", "Lipstick", "Foundation", "Face Serum"
    ],

    "Grocery": [
        "Rice", "Wheat Flour", "Sugar", "Salt", "Cooking Oil", "Coffee",
        "Tea", "Milk", "Bread", "Biscuits", "Chocolate", "Cereal"
    ],

    "Sports": [
        "Cricket Bat", "Cricket Ball", "Football", "Basketball", "Tennis Racket",
        "Badminton Racket", "Yoga Mat", "Dumbbells", "Treadmill", "Cycling Helmet"
    ],

    "Furniture": [
        "Office Chair", "Gaming Chair", "Sofa", "Dining Table", "Coffee Table",
        "Bed", "Wardrobe", "Bookshelf", "Study Table", "TV Stand"
    ]
}

products = []
product_number = 1001

for category, names in product_categories.items():
    for name in names:
        products.append({
            "product_id": f"P{product_number}",
            "product_name": name,
            "category": category,
            "subcategory": "General",
            "base_price": round(random.uniform(100, 100000), 2)
        })
        product_number += 1

print(f"Products available: {len(products)}")

In [0]:
SOURCE_CONFIG = {
    "POS": {
        "channel": "Store",
        "payments": ["Cash", "UPI", "Credit Card", "Debit Card"],
        "store_prefix": "POS"
    },

    "E_COMMERCE": {
        "channel": "Online",
        "payments": ["UPI", "Credit Card", "Debit Card", "Net Banking", "Wallet"],
        "store_prefix": "WEB"
    },

    "MOBILE_APP": {
        "channel": "Mobile App",
        "payments": ["UPI", "Credit Card", "Wallet", "Net Banking"],
        "store_prefix": "APP"
    },

    "RETAIL_STORE": {
        "channel": "Store",
        "payments": ["Cash", "UPI", "Credit Card", "Debit Card"],
        "store_prefix": "RTL"
    },

    "MARKETPLACE": {
        "channel": "Marketplace",
        "payments": ["UPI", "Credit Card", "Debit Card", "Wallet", "Net Banking"],
        "store_prefix": "MKT"
    }
}

In [0]:
def generate_record(i, source_name):
    config = SOURCE_CONFIG[source_name]

    country, region, city, state, zip_code = random.choice(locations)
    product = random.choice(products)

    sale_timestamp = (
        RUN_TIMESTAMP -
        timedelta(minutes=random.randint(0, 60 * 24 * 30))
    )

    quantity = random.randint(1, 5)
    unit_price = product["base_price"]

    discount = random.choice([0, 5, 10, 15, 20])
    tax = random.choice([5, 12, 18])

    subtotal = quantity * unit_price
    discount_amount = subtotal * discount / 100
    taxable_amount = subtotal - discount_amount
    tax_amount = taxable_amount * tax / 100

    total_amount = round(
        subtotal - discount_amount + tax_amount,
        2
    )

    cost_price = round(
        unit_price * random.uniform(0.50, 0.85),
        2
    )

    profit_amount = round(
        total_amount - (cost_price * quantity),
        2
    )

    order_status = random.choice([
        "Completed",
        "Completed",
        "Completed",
        "Cancelled",
        "Returned"
    ])

    return_flag = order_status == "Returned"
    return_quantity = quantity if return_flag else 0

    # Source-specific sales ID.
    # sale_id = f"{config['store_prefix']}{i:06d}"
    sale_id = f"{config['store_prefix']}{RUN_ID[-4:]}{i:06d}"

    record = {
        "sale_id": sale_id,

        "sale_timestamp": sale_timestamp.strftime("%Y-%m-%d %H:%M:%S"),
        "sale_date": sale_timestamp.strftime("%Y-%m-%d"),

        "country": country,
        "region": region,
        "state": state,
        "city": city,
        "zip_code": zip_code,

        "store_id": f"ST{random.randint(1, 50):03d}",
        "store_name": f"FreshMart {city}",

        "product_id": product["product_id"],
        "product_name": product["product_name"],
        "category": product["category"],
        "subcategory": product["subcategory"],

        "quantity": quantity,
        "unit_price": unit_price,
        "discount": float(discount),
        "tax": float(tax),
        "total_amount": total_amount,

        "payment_method": random.choice(config["payments"]),

        "customer_id": f"C{random.randint(100000, 999999)}",
        "customer_type": random.choice([
            "New", "Regular", "Premium", "VIP"
        ]),

        "currency": random.choice(["INR", "USD", "EUR", "GBP"]),
        "exchange_rate": round(random.uniform(0.5, 100), 4),

        "sales_channel": config["channel"],
        "order_status": order_status,

        "return_flag": return_flag,
        "return_quantity": return_quantity,

        "shipping_cost": round(random.uniform(50, 1000), 2),
        "cost_price": cost_price,
        "profit_amount": profit_amount,

        "salesperson_id": f"EMP{random.randint(1000, 9999)}",
        "promotion_id": f"PROMO{random.randint(100, 999)}",

        "coupon_code": random.choice([
            "SAVE10", "NEW20", "FESTIVE15", None
        ]),

        "inventory_before_sale": random.randint(10, 500),
        "inventory_after_sale": random.randint(1, 500),

        "weather_condition": random.choice([
            "Sunny", "Cloudy", "Rainy", "Stormy", "Clear"
        ]),

        "temperature_c": round(random.uniform(5, 42), 2),

        "data_source": source_name,

        "ingestion_timestamp": RUN_TIMESTAMP.strftime(
            "%Y-%m-%d %H:%M:%S"
        ),

        "run_id": RUN_ID
    }

    return record

In [0]:
bad_cases = {
    0: {"quantity": -2},
    1: {"unit_price": None},
    2: {"city": "Delhii"},
    3: {"country": "Indai"},
    4: {"payment_method": "upi "},
    5: {"category": "electronic"},
    6: {"discount": 150},
    7: {"tax": -5},
    8: {"total_amount": 9999999.99},
    9: {"store_id": None},
    10: {"product_id": "P9999"},
    11: {"order_status": "Unknown"},
    12: {"currency": "XYZ"},
    13: {"shipping_cost": -500},
    14: {
        "return_flag": True,
        "return_quantity": 99
    },
    15: {"customer_id": None},
    16: {"zip_code": "INVALID"},
    17: {"customer_type": "Unknown"},
    18: {"cost_price": -1000},
    19: {"sales_channel": "UnknownChannel"},
    20: {"temperature_c": 999},
    21: {"exchange_rate": 0},
    22: {"inventory_after_sale": 999999},
    23: {"weather_condition": "UnknownWeather"},
    24: {"salesperson_id": None},
    25: {"promotion_id": ""},
    26: {"region": "UnknownRegion"},
    27: {"state": "UnknownState"},
    28: {"store_name": None},
    29: {"product_name": None},
    30: {"tax": 250},
    31: {"quantity": 0},
    32: {"discount": -10},
    33: {"return_quantity": -5},
    34: {"coupon_code": "INVALID_COUPON"},
    35: {"country": None},
    36: {"total_amount": None},
    37: {"data_source": "UNKNOWN_SOURCE"},
    38: {"sale_timestamp": "2035-01-01 00:00:00"},
    39: {"sale_date": "INVALID_DATE"}
}

In [0]:
all_audit_rows = []
source_summary = []

for source_name in SOURCE_SYSTEMS:

    print(f"Generating source: {source_name}")

    records = [
        generate_record(i, source_name)
        for i in range(1, NUM_RECORDS + 1)
    ]

    # Apply intentional data-quality issues.
    for index, changes in bad_cases.items():
        records[index].update(changes)

    # JSON Lines format:
    # one JSON object per line.
    json_content = "\n".join(
        json.dumps(
            record,
            ensure_ascii=False
        )
        for record in records
    )

    source_path = (
        f"{RAW_PATH}/{source_name.lower()}/sales.json"
    )

    dbutils.fs.mkdirs(
        f"{RAW_PATH}/{source_name.lower()}"
    )

    # Each source is independently overwritten on every run.
    dbutils.fs.put(
        source_path,
        json_content,
        overwrite=True
    )

    # Add every generated record to the audit collection.
    for record in records:
        record_copy = record.copy()

        record_copy["audit_timestamp"] = (
            RUN_TIMESTAMP.strftime("%Y-%m-%d %H:%M:%S")
        )

        all_audit_rows.append(record_copy)

    source_summary.append({
        "source": source_name,
        "records": len(records),
        "bad_records": len(bad_cases),
        "path": source_path
    })

    print(
        f"Completed: {source_name} | "
        f"Records: {len(records)} | "
        f"Bad records: {len(bad_cases)}"
    )

In [0]:
string_schema = StructType([
    StructField("sale_id", StringType(), True),
    StructField("sale_timestamp", StringType(), True),
    StructField("sale_date", StringType(), True),
    StructField("country", StringType(), True),
    StructField("region", StringType(), True),
    StructField("state", StringType(), True),
    StructField("city", StringType(), True),
    StructField("zip_code", StringType(), True),
    StructField("store_id", StringType(), True),
    StructField("store_name", StringType(), True),
    StructField("product_id", StringType(), True),
    StructField("product_name", StringType(), True),
    StructField("category", StringType(), True),
    StructField("subcategory", StringType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("unit_price", DoubleType(), True),
    StructField("discount", DoubleType(), True),
    StructField("tax", DoubleType(), True),
    StructField("total_amount", DoubleType(), True),
    StructField("payment_method", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("customer_type", StringType(), True),
    StructField("currency", StringType(), True),
    StructField("exchange_rate", DoubleType(), True),
    StructField("sales_channel", StringType(), True),
    StructField("order_status", StringType(), True),
    StructField("return_flag", BooleanType(), True),
    StructField("return_quantity", IntegerType(), True),
    StructField("shipping_cost", DoubleType(), True),
    StructField("cost_price", DoubleType(), True),
    StructField("profit_amount", DoubleType(), True),
    StructField("salesperson_id", StringType(), True),
    StructField("promotion_id", StringType(), True),
    StructField("coupon_code", StringType(), True),
    StructField("inventory_before_sale", IntegerType(), True),
    StructField("inventory_after_sale", IntegerType(), True),
    StructField("weather_condition", StringType(), True),
    StructField("temperature_c", DoubleType(), True),
    StructField("data_source", StringType(), True),
    StructField("ingestion_timestamp", StringType(), True),
    StructField("run_id", StringType(), True),
    StructField("audit_timestamp", StringType(), True)
])

audit_df = spark.createDataFrame(
    all_audit_rows,
    schema=string_schema
)

audit_df = (
    audit_df
    .withColumn(
        "sale_timestamp",
        F.expr("try_cast(sale_timestamp as timestamp)")
    )
    .withColumn(
        "sale_date",
        F.expr("try_cast(sale_date as date)")
    )
    .withColumn(
        "ingestion_timestamp",
        F.expr("try_cast(ingestion_timestamp as timestamp)")
    )
    .withColumn(
        "audit_timestamp",
        F.expr("try_cast(audit_timestamp as timestamp)")
    )
)

audit_df.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable(AUDIT_TABLE)

In [0]:
print("=" * 70)
print("MULTI-SOURCE SALES GENERATION COMPLETED")
print("=" * 70)

print(f"Run ID              : {RUN_ID}")
print(f"Sources              : {len(SOURCE_SYSTEMS)}")
print(f"Records per source   : {NUM_RECORDS}")
print(f"Total records        : {len(SOURCE_SYSTEMS) * NUM_RECORDS}")
print(f"Bad records/source   : {len(bad_cases)}")
print(f"Total bad records    : {len(SOURCE_SYSTEMS) * len(bad_cases)}")
print(f"Raw root path        : {RAW_PATH}")
print(f"Audit table          : {AUDIT_TABLE}")

print("-" * 70)

for item in source_summary:
    print(
        f"{item['source']:15} | "
        f"{item['records']:4} records | "
        f"{item['bad_records']:2} bad records | "
        f"{item['path']}"
    )

print("=" * 70)